# الدفتر الرابع — اختبار التكرار على ورم دبقي منخفض الدرجة (LGG)

**الهدف:** نفس اللوحة بلا أي تعديل، على ورم آخر وبعيّنة أكبر (نحو 500 مريض). إن كان لـ Vmem أثر حقيقي فهذا أفضل مكان لرؤيته.

**التشغيل:** Runtime ← Run all (5–10 دقائق). لا تعدّل شيئاً.

**ملاحظة:** نفس المعايير الأربعة المُعلنة مسبقاً. لا تغيير بعد رؤية النتيجة.


In [ ]:
!pip install lifelines requests pandas matplotlib -q
print("تم التثبيت")

In [ ]:
import numpy as np, pandas as pd

R, F, T_BODY = 8.314, 96485, 310.15
CONC = dict(K_out=5.0, K_in=140.0, Na_out=145.0, Na_in=15.0, Cl_in=10.0, Cl_out=110.0)
BASELINE = dict(p_k=1.0, p_na=0.02, p_cl=0.10)
RTF = (R * T_BODY) / F * 1000.0

def ghk_vmem(p_k, p_na, p_cl):
    num = p_k * CONC["K_out"] + p_na * CONC["Na_out"] + p_cl * CONC["Cl_in"]
    den = p_k * CONC["K_in"] + p_na * CONC["Na_in"] + p_cl * CONC["Cl_out"]
    return RTF * np.log(num / den)

def scale_matrix(expr, reference, max_scale=5.0):
    """نفس منطق log2 السابق لكن لكل المرضى دفعة واحدة. القيم الناقصة = 1."""
    s = np.log2(np.clip(expr, 1e-6, None) + 1) / np.log2(reference + 1)
    s = np.clip(s, 1.0 / max_scale, max_scale)
    return np.where(np.isnan(expr), 1.0, s)

def vmem_from_roles(expr_df, roles):
    """
    expr_df: صفوف = مرضى، أعمدة = جينات
    roles: {"K": [جينات], "Na": [جينات], "Cl": [جينات]}
    """
    ref = expr_df.mean(axis=0)
    agg = {}
    for ch, genes in roles.items():
        if genes:
            m = scale_matrix(expr_df[genes].to_numpy(dtype=float), ref[genes].to_numpy(dtype=float))
            agg[ch] = m.mean(axis=1)
        else:
            agg[ch] = np.ones(len(expr_df))
    v = ghk_vmem(BASELINE["p_k"] * agg["K"], BASELINE["p_na"] * agg["Na"], BASELINE["p_cl"] * agg["Cl"])
    return pd.Series(v, index=expr_df.index, name="vmem")

print("خط الأساس:", round(float(ghk_vmem(1.0, 0.02, 0.10)), 2), "mV (المتوقع ≈ -75.7)")

In [ ]:
import requests

API = "https://www.cbioportal.org/api"
STUDY = "lgg_tcga_pan_can_atlas_2018"
SAMPLE_LIST = f"{STUDY}_rna_seq_v2_mrna"
PROFILE = f"{STUDY}_rna_seq_v2_mrna"

ROLES = {"Na": ["SCN9A"], "K": ["KCNJ10", "KCNJ2"], "Cl": ["CLIC1"]}   # نفس الجينات السابقة
OUR_GENES = [g for gs in ROLES.values() for g in gs]

def entrez_map(symbols):
    r = requests.post(f"{API}/genes/fetch", params={"geneIdType": "HUGO_GENE_SYMBOL"},
                      json=symbols, timeout=30)
    r.raise_for_status()
    return {g["hugoGeneSymbol"]: g["entrezGeneId"] for g in r.json()}

def fetch_molecular(entrez_ids):
    """يرجع DataFrame: صفوف = sampleId، أعمدة = entrezGeneId."""
    r = requests.post(f"{API}/molecular-profiles/{PROFILE}/molecular-data/fetch",
                      params={"projection": "SUMMARY"},
                      json={"entrezGeneIds": [int(i) for i in entrez_ids], "sampleListId": SAMPLE_LIST},
                      timeout=120)
    r.raise_for_status()
    rec = pd.DataFrame(r.json())
    if rec.empty:
        return pd.DataFrame()
    rec["value"] = pd.to_numeric(rec["value"], errors="coerce")
    return rec.pivot_table(index="sampleId", columns="entrezGeneId", values="value", aggfunc="first")

def to_patient_index(df):
    out = df.copy()
    out.index = [i[:12] for i in out.index]
    return out[~out.index.duplicated(keep="first")]


# ===== لوحة الجينات المُعلنة مسبقاً =====
PANEL = {
    "Na": ["SCN9A", "SCN1A", "SCN2A", "SCN3A", "SCN8A"],
    "K":  ["KCNJ10", "KCNJ2", "KCNJ4", "KCNJ11", "KCNK1", "KCNK3", "KCNMA1", "KCNN4", "KCNH2", "KCNQ1"],
    "Cl": ["CLIC1", "CLIC4", "CLCN2", "CLCN3", "ANO1", "CFTR"],
}
ALL = [g for gs in PANEL.values() for g in gs]
em = entrez_map(ALL)
missing = [g for g in ALL if g not in em]
if missing: print("جينات غير موجودة في الواجهة (تُتجاوز):", missing)
inv = {v: k for k, v in em.items()}
expr_all = to_patient_index(fetch_molecular(list(em.values())).rename(columns=inv))

# شرط التعبير الوحيد: متوسط > 20 وأقل من 5% قيم ناقصة
ok = [g for g in expr_all.columns if expr_all[g].mean() > 20 and expr_all[g].notna().mean() > 0.95]
dropped = [g for g in expr_all.columns if g not in ok]
print("جينات مستبعدة لضعف التعبير:", dropped)
ROLES = {ch: [g for g in gs if g in ok] for ch, gs in PANEL.items()}
for ch, gs in ROLES.items(): print(f"{ch}: {len(gs)} جين -> {gs}")
expr_p = expr_all[ok]

# بقاء + عمر
r = requests.get(f"{API}/studies/{STUDY}/clinical-data",
                 params={"clinicalDataType": "PATIENT", "projection": "SUMMARY"}, timeout=60)
r.raise_for_status()
cl = pd.DataFrame(r.json())
cl = cl[cl["clinicalAttributeId"].isin(["OS_STATUS", "OS_MONTHS", "AGE"])]
surv_df = cl.pivot_table(index="patientId", columns="clinicalAttributeId", values="value", aggfunc="first")
surv_df["OS_MONTHS"] = pd.to_numeric(surv_df["OS_MONTHS"], errors="coerce")
surv_df["event"] = surv_df["OS_STATUS"].astype(str).str.startswith("1").astype(int)
surv_df["AGE"] = pd.to_numeric(surv_df.get("AGE"), errors="coerce")

# مجموعة عشوائية للمقارنة
def get_all_entrez_ids():
    ids = []
    try:
        for page in range(0, 10):
            rr = requests.get(f"{API}/genes", params={"projection": "SUMMARY", "pageSize": 10000, "pageNumber": page}, timeout=60)
            rr.raise_for_status(); chunk = rr.json()
            if not chunk: break
            ids += [x["entrezGeneId"] for x in chunk if x.get("type", "protein-coding") == "protein-coding"]
    except Exception as e:
        print("تعذّر GET /genes، بديل:", repr(e)[:80])
    if len(ids) < 2000: ids = list(range(1, 100000))
    return ids
rng0 = np.random.default_rng(2026)
all_ids = np.array(sorted(set(get_all_entrez_ids()) - set(em.values())))
sample_ids = rng0.choice(all_ids, size=min(7000, len(all_ids)), replace=False)
parts = []
for i in range(0, len(sample_ids), 400):
    try:
        p_ = fetch_molecular(sample_ids[i:i+400])
        if not p_.empty: parts.append(p_)
    except Exception as e:
        print("دفعة فاشلة (تُتجاوز):", repr(e)[:70])
pool = to_patient_index(pd.concat(parts, axis=1)) if parts else pd.DataFrame()
keep = (pool.notna().mean() > 0.95) & (pool.mean() > 20)
pool = pool.loc[:, keep]
print(f"مجموعة عشوائية: {pool.shape[1]} جيناً × {pool.shape[0]} مريض")
print("المرضى الذين لديهم تعبير + عمر + بقاء:", int(surv_df.join(expr_p.iloc[:, :1], how="inner")[["OS_MONTHS","AGE"]].dropna().shape[0]))

## 1) النتيجة الأساسية: Vmem من اللوحة الموسّعة

In [ ]:
from lifelines import CoxPHFitter
import matplotlib; import matplotlib.pyplot as plt

def make_data(x, extra=None):
    d = surv_df.join(x.rename("x"), how="inner")
    return d[["OS_MONTHS", "event", "x"] + (extra or [])].dropna()

def cox_fit(d):
    c = CoxPHFitter(); c.fit(d, duration_col="OS_MONTHS", event_col="event")
    return c.summary.loc["x"]

roles_ok = {ch: list(ROLES.get(ch, [])) for ch in ("Na", "K", "Cl")}
vmem = vmem_from_roles(expr_p, roles_ok)

d_all = make_data(vmem)
d_age = make_data(vmem, extra=["AGE"])
s_all = cox_fit(d_all); s_age_base = cox_fit(d_age[["OS_MONTHS","event","x"]]); s_age = cox_fit(d_age)
sd = d_all["x"].std()
print(f"Vmem: SD = {sd:.2f} mV")
print(f"[1] كل المرضى (n={len(d_all)}) بلا عمر:     HR/mV={s_all['exp(coef)']:.3f}  p={s_all['p']:.4f}")
print(f"[2] مرضى لديهم عمر (n={len(d_age)}) بلا عمر: HR/mV={s_age_base['exp(coef)']:.3f}  p={s_age_base['p']:.4f}")
print(f"[3] نفس المرضى مع ضبط العمر (الأساسي):    HR/mV={s_age['exp(coef)']:.3f}  p={s_age['p']:.4f}")
PRIMARY_P = float(s_age["p"]); PRIMARY_COEF = float(s_age["coef"]); PRIMARY_Z = float(s_age["z"])

## 2) هل يعتمد على جين واحد؟ (حذف جين جين + حذف أيون كامل)

In [ ]:
rows = []
for g in ok:
    rl = {ch: [x for x in gs if x != g] for ch, gs in roles_ok.items()}
    v = vmem_from_roles(expr_p, rl)
    s = cox_fit(make_data(v, extra=["AGE"]))
    rows.append((f"بدون {g}", s["exp(coef)"], s["p"]))
for ch in ("Na", "K", "Cl"):
    rl = {c: ([] if c == ch else gs) for c, gs in roles_ok.items()}
    v = vmem_from_roles(expr_p, rl)
    s = cox_fit(make_data(v, extra=["AGE"]))
    rows.append((f"بدون كل جينات {ch}", s["exp(coef)"], s["p"]))
loo = pd.DataFrame(rows, columns=["test", "HR/mV", "p"]).round(4)
print(loo.to_string(index=False))
LOO_MAXP = float(loo[loo.test.str.startswith("بدون ") & ~loo.test.str.contains("كل")]["p"].max())

## 3) استقرار الأثر: إعادة سحب المرضى 1000 مرة (Bootstrap)

In [ ]:
rngb = np.random.default_rng(11)
coefs = []
for _ in range(1000):
    idx = rngb.integers(0, len(d_age), len(d_age))
    try:
        c = CoxPHFitter(); c.fit(d_age.iloc[idx].reset_index(drop=True), duration_col="OS_MONTHS", event_col="event")
        coefs.append(c.params_["x"])
    except Exception:
        pass
coefs = np.array(coefs)
lo, hi = np.percentile(coefs, [2.5, 97.5])
print(f"HR/mV (بعد ضبط العمر) مجال الثقة 95%: [{np.exp(lo):.3f} ، {np.exp(hi):.3f}]")
print(f"نسبة المرات التي كان فيها الاتجاه موجباً (كفرضية ليفين): {(coefs > 0).mean()*100:.1f}%")
BOOT_EXCL1 = (np.exp(lo) > 1) or (np.exp(hi) < 1)

## 4) مقارنة بـ 1000 مجموعة جينات عشوائية (بنفس البنية، وبعد ضبط العمر)

In [ ]:
common = d_age.index.intersection(pool.index)
dd = d_age.loc[common]; pl = pool.loc[common]
n_roles = {ch: len(gs) for ch, gs in roles_ok.items()}
genes = list(pl.columns); rngn = np.random.default_rng(7)
zs = []
for _ in range(1000):
    pick = rngn.choice(genes, size=sum(n_roles.values()), replace=False)
    a = n_roles["Na"]; b = a + n_roles["K"]
    rl = {"Na": list(pick[:a]), "K": list(pick[a:b]), "Cl": list(pick[b:])}
    v = vmem_from_roles(pl, rl)
    d = dd[["OS_MONTHS", "event", "AGE"]].copy(); d["x"] = v.loc[d.index]
    try:
        zs.append(cox_fit(d)["z"])
    except Exception:
        pass
zs = np.array(zs)
p_two = (1 + (np.abs(zs) >= abs(PRIMARY_Z)).sum()) / (1 + len(zs))
p_one = (1 + (zs >= PRIMARY_Z).sum()) / (1 + len(zs))
print(f"z لجيناتنا = {PRIMARY_Z:.2f} | محاكاة ناجحة: {len(zs)}")
print(f"p تجريبي (اتجاهان) = {p_two:.4f} | (اتجاه ليفين) = {p_one:.4f}")
plt.figure(figsize=(7, 4.5)); plt.hist(zs, bins=40, color="#9ecae1", edgecolor="white")
plt.axvline(PRIMARY_Z, color="crimson", lw=2, label=f"our panel (z={PRIMARY_Z:.2f})")
plt.title(f"Random gene sets vs ours, age-adjusted (empirical p = {p_two:.3f})")
plt.xlabel("Cox z-score of predicted Vmem"); plt.ylabel("Random gene sets"); plt.legend()
plt.tight_layout(); plt.savefig("null_panel.png", dpi=150); plt.show()

## 5) الحكم النهائي

In [ ]:
print("=" * 52); print("الخلاصة (المعيار مُعلن مسبقاً)"); print("=" * 52)
n_genes = len(ok)
print(f"عدد الجينات المستخدمة: {n_genes}  | مرضى التحليل الأساسي: {len(d_age)}\n")
c1 = PRIMARY_P < 0.05;  c2 = p_two < 0.05;  c3 = BOOT_EXCL1;  c4 = LOO_MAXP < 0.10
print(("✅" if c1 else "⚠️"), f"1) بعد ضبط العمر: p = {PRIMARY_P:.4f}")
print(("✅" if c2 else "⚠️"), f"2) أفضل من الجينات العشوائية: p تجريبي = {p_two:.4f}")
print(("✅" if c3 else "⚠️"), "3) مجال ثقة Bootstrap لا يشمل 1")
print(("✅" if c4 else "⚠️"), f"4) لا يعتمد على جين واحد (أسوأ p عند حذف جين = {LOO_MAXP:.3f})")
k = sum([c1, c2, c3, c4]); print()
if k == 4: print("🟢 صمدت في الاختبارات الأربعة: إشارة جديرة بالتحقق في مجموعة مستقلة. (ليست إثباتاً.)")
elif k >= 2: print("🟡 صمدت جزئياً: إشارة أولية ضعيفة. نحتاج مجموعة مستقلة قبل أي ادعاء.")
else: print("🔴 لم تصمد: لا دليل كافٍ على أن Vmem المتنبَّأ به يتنبأ بالبقاء في هذه البيانات. وهذه نتيجة علمية صالحة للتوثيق.")